# Aquifer Petrignano: KFold vs TimeSeriesSplit

This notebook is for the Petrignano aquifer exercise. I use the previous two months of data to predict the current month value of `Depth_to_Groundwater_P25` (DP25).

The comparison I care about is simple: a normal shuffled `KFold` score versus a time-aware `TimeSeriesSplit` score. Since this is time-series data, the shuffled version can look better than it really is.

The data file should be named `petrignanos.csv` and kept in the same folder as this notebook. If I run it in Colab without the file already there, the notebook asks me to upload it.


## AI Use Note

I used AI help to build the first version of the code. The code cells keep `prompt` comments at the top so the AI use is documented for submission. I reviewed and changed the code before using it.


In [ ]:
# prompt: Help me set up the imports for pandas, plotting, cross-validation, grid search, and a decision tree model.
# prompt changes: I kept the imports to common packages that work in Colab.

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, TimeSeriesSplit, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
RANDOM_STATE = 42

In [ ]:
# prompt: Help me load petrignanos.csv from this folder, nearby data folders, or a Colab upload.
# prompt changes: I kept a simple uploader fallback and left a place to add my own raw GitHub CSV URL later.

RAW_GITHUB_DATA_URLS = [
    # Add your raw GitHub CSV URL here after creating a separate Aquifer repository.
]

def load_aquifer_csv(filenames=("petrignanos.csv", "Aquifer_Petrignano.csv")):
    candidates = [
        folder / filename
        for filename in filenames
        for folder in [
            Path("."),
            Path("01_Data"),
            Path("../01_Data"),
            Path(".."),
            Path("../../01_Data"),
            Path("../.."),
            Path("/content"),
            Path("/content/01_Data"),
        ]
    ]
    for path in candidates:
        if path.exists():
            return pd.read_csv(path), str(path)

    for url in RAW_GITHUB_DATA_URLS:
        try:
            return pd.read_csv(url), url
        except Exception:
            pass

    try:
        from google.colab import files
        print("Upload the Aquifer Petrignano CSV from Moodle Materials.")
        uploaded = files.upload()
        for filename in filenames:
            if filename in uploaded:
                return pd.read_csv(filename), filename
        if uploaded:
            uploaded_filename = next(iter(uploaded.keys()))
            return pd.read_csv(uploaded_filename), uploaded_filename
    except ModuleNotFoundError:
        pass

    raise FileNotFoundError(
        f"Could not find any of {filenames}. Put the CSV in the notebook folder, repo root, 01_Data/, "
        "or run this notebook in Colab and upload it when prompted."
    )

raw, csv_source = load_aquifer_csv()

print(f"Loaded: {csv_source}")
print(raw.shape)
display(raw.head())

## 0. Data Setup

The original data is daily, but the assignment asks for a monthly prediction. I first clean the dates and missing values, then aggregate to month level. Rainfall and volume are summed by month. The other numeric variables are averaged.

After that I make lagged columns for month `t-1` and month `t-2`. Those lagged columns are the predictors. The response variable is the current month DP25 value.


In [ ]:
# prompt: Help me clean the daily Petrignano data, aggregate it monthly, add seasonality, and create two months of lagged predictors.
# prompt changes: I treated rainfall zeroes as valid, filled missing values forward after measurements started, and dropped early incomplete months.

TARGET = "Depth_to_Groundwater_P25"
DATE_COL = "Date"

if DATE_COL not in raw.columns:
    date_matches = [c for c in raw.columns if c.lower() == "date"]
    if date_matches:
        DATE_COL = date_matches[0]
    else:
        raise KeyError("The dataset must contain a Date column.")

if TARGET not in raw.columns:
    raise KeyError(f"Expected response variable {TARGET!r}. Available columns: {list(raw.columns)}")

df = raw.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL], dayfirst=True, errors="coerce")
df = df.dropna(subset=[DATE_COL]).sort_values(DATE_COL).drop_duplicates(subset=[DATE_COL])

numeric_cols = [c for c in df.columns if c != DATE_COL]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

rain_cols = [c for c in numeric_cols if "rainfall" in c.lower() or "rain" in c.lower()]
zero_as_missing_cols = [
    c for c in numeric_cols
    if c not in rain_cols and any(token in c.lower() for token in ["depth", "groundwater", "hydrometry", "volume"])
]
df[zero_as_missing_cols] = df[zero_as_missing_cols].replace(0, np.nan)

df = df.set_index(DATE_COL).asfreq("D")
df[numeric_cols] = df[numeric_cols].ffill()

volume_cols = [c for c in numeric_cols if "volume" in c.lower()]
sum_cols = set(rain_cols + volume_cols)
def monthly_sum(series):
    return series.sum(min_count=1)

agg_map = {c: monthly_sum if c in sum_cols else "mean" for c in numeric_cols}
monthly = df.resample("MS").agg(agg_map).sort_index()

month_number = monthly.index.month
monthly["month_sin"] = np.sin(2 * np.pi * month_number / 12)
monthly["month_cos"] = np.cos(2 * np.pi * month_number / 12)

lag_source_cols = [c for c in monthly.columns if c not in ["month_sin", "month_cos"]]
lagged_parts = []
for lag in [1, 2]:
    lagged = monthly[lag_source_cols].shift(lag).add_suffix(f"_lag{lag}")
    lagged_parts.append(lagged)

model_data = pd.concat(
    [monthly[[TARGET, "month_sin", "month_cos"]]] + lagged_parts,
    axis=1
).dropna()

y = model_data[TARGET]
X = model_data.drop(columns=[TARGET])

print("Daily rows after date cleaning:", len(df))
print("Monthly rows available:", len(monthly))
print("Supervised rows after 2-month lagging:", len(model_data))
print("Response variable:", TARGET)
print("Number of predictors:", X.shape[1])
print("Modeling period:", model_data.index.min().strftime("%Y-%m"), "to", model_data.index.max().strftime("%Y-%m"))

display(X.head())
display(y.head())

In [ ]:
# prompt: Help me plot monthly DP25 so I can see the target over time.
# prompt changes: I kept the plot simple because it is only for checking the series.

fig, ax = plt.subplots(figsize=(12, 4))
monthly[TARGET].plot(ax=ax, color="#1f77b4", linewidth=1.5)
ax.set_title("Monthly Depth_to_Groundwater_P25")
ax.set_xlabel("Month")
ax.set_ylabel("Depth to groundwater")
ax.grid(True, alpha=0.25)
plt.show()

## 1. Hard Split

I keep the data in date order and hold out the last 20% of the months as the future test set. This is the independent test data set.


In [ ]:
# prompt: Help me split X and y chronologically with the last 20 percent held out for testing.
# prompt changes: I printed the date ranges so I know exactly which months are train and test.

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

train_start, train_end = X_train_full.index.min(), X_train_full.index.max()
test_start, test_end = X_test.index.min(), X_test.index.max()

print(f"Training months: {train_start:%Y-%m} to {train_end:%Y-%m} ({len(X_train_full)} rows)")
print(f"Independent test months: {test_start:%Y-%m} to {test_end:%Y-%m} ({len(X_test)} rows)")

## 2. Cross-Validation Choices

Here I set up the two validation methods. `KFold` is shuffled, so it ignores time order. `TimeSeriesSplit` keeps the folds in chronological order.


In [ ]:
# prompt: Help me define shuffled KFold and TimeSeriesSplit for this comparison.
# prompt changes: I used the exact split settings from the assignment.

cv_naive = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_temporal = TimeSeriesSplit(n_splits=5)

print(cv_naive)
print(cv_temporal)

## 3. Fixed Model

This section uses one decision tree setup so the only thing changing is the cross-validation method.


In [ ]:
# prompt: Help me compare the two CV methods with one fixed decision tree pipeline.
# prompt changes: I kept R2 as the main metric and added MAE/RMSE only as extra context.

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE))
])

scores_naive = cross_val_score(pipe, X_train_full, y_train_full, cv=cv_naive, scoring="r2")
scores_temporal = cross_val_score(pipe, X_train_full, y_train_full, cv=cv_temporal, scoring="r2")

print(f"Naive CV R2:    {scores_naive.mean():.4f} (+/- {scores_naive.std():.4f})")
print(f"Temporal CV R2: {scores_temporal.mean():.4f} (+/- {scores_temporal.std():.4f})")

pipe.fit(X_train_full, y_train_full)
fixed_pred = pipe.predict(X_test)
final_test_r2 = r2_score(y_test, fixed_pred)
fixed_mae = mean_absolute_error(y_test, fixed_pred)
fixed_rmse = np.sqrt(mean_squared_error(y_test, fixed_pred))

print(f"\nActual Test R2 on unseen future months ({test_start:%Y-%m} to {test_end:%Y-%m}): {final_test_r2:.4f}")
print(f"Actual Test MAE: {fixed_mae:.4f}")
print(f"Actual Test RMSE: {fixed_rmse:.4f}")

## 4. Model Selection

Now I use `GridSearchCV` to tune the tree. I run the same grid twice: once with shuffled KFold and once with TimeSeriesSplit.


In [ ]:
# prompt: Help me complete the GridSearchCV function using a scaler, decision tree, and several max_depth values.
# prompt changes: I also tuned min_samples_leaf and used n_jobs=1 so it runs reliably on Windows and Colab.

def evaluate_model_selection(X_train, y_train, X_test, y_test, cv_strategy, name):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("regressor", DecisionTreeRegressor(random_state=RANDOM_STATE))
    ])

    param_grid = {
        "regressor__max_depth": [2, 4, 6, 8, 10, 12, None],
        "regressor__min_samples_leaf": [1, 3, 5, 10]
    }

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=cv_strategy,
        scoring="r2",
        n_jobs=1,
        return_train_score=True
    )

    grid.fit(X_train, y_train)

    y_pred = grid.predict(X_test)
    test_r2 = r2_score(y_test, y_pred)
    test_mae = mean_absolute_error(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    print(f"\n===== Results for: {name} =====")
    print(f"Best Parameters found: {grid.best_params_}")
    print(f"Internal CV Score (R2): {grid.best_score_:.4f}")
    print(f"Independent Test Score (R2): {test_r2:.4f}")
    print(f"Independent Test MAE: {test_mae:.4f}")
    print(f"Independent Test RMSE: {test_rmse:.4f}")

    return {
        "name": name,
        "grid": grid,
        "best_params": grid.best_params_,
        "cv_r2": grid.best_score_,
        "test_r2": test_r2,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_predictions": pd.Series(y_pred, index=y_test.index, name=f"{name} prediction")
    }

## 5. Compare Results

The important part is the gap between the internal CV score and the score on the held-out future months.


In [ ]:
# prompt: Help me run model selection for KFold and TimeSeriesSplit and compare CV score to test score.
# prompt changes: I added a small table showing the gap between CV and test R2.

result_naive = evaluate_model_selection(
    X_train_full, y_train_full, X_test, y_test, cv_naive, "Naive K-Fold"
)
result_temporal = evaluate_model_selection(
    X_train_full, y_train_full, X_test, y_test, cv_temporal, "Temporal Split"
)

comparison = pd.DataFrame([
    {
        "strategy": result_naive["name"],
        "internal_cv_r2": result_naive["cv_r2"],
        "independent_test_r2": result_naive["test_r2"],
        "cv_minus_test_gap": result_naive["cv_r2"] - result_naive["test_r2"],
        "best_params": result_naive["best_params"],
    },
    {
        "strategy": result_temporal["name"],
        "internal_cv_r2": result_temporal["cv_r2"],
        "independent_test_r2": result_temporal["test_r2"],
        "cv_minus_test_gap": result_temporal["cv_r2"] - result_temporal["test_r2"],
        "best_params": result_temporal["best_params"],
    }
])

display(comparison)

In [ ]:
# prompt: Help me plot observed DP25 against both model predictions on the test months.
# prompt changes: I kept this as one simple comparison chart for the video.

plot_df = pd.concat(
    [
        y_test.rename("Observed"),
        result_naive["test_predictions"],
        result_temporal["test_predictions"],
    ],
    axis=1
)

fig, ax = plt.subplots(figsize=(12, 5))
plot_df.plot(ax=ax, linewidth=1.8)
ax.set_title("Independent Test Data Set: Observed vs Predicted DP25")
ax.set_xlabel("Month")
ax.set_ylabel("Depth_to_Groundwater_P25")
ax.grid(True, alpha=0.25)
plt.show()

display(plot_df.head())

## Notes For My Video

Use the numbers printed above. The points I need to cover are:

- **response variable**: `Depth_to_Groundwater_P25`, also called DP25.
- **predictors**: all variables from the previous two months, plus the month sine/cosine columns.
- **imputation**: missing values are filled forward after each variable starts; early incomplete months are dropped.
- **cross-validation** and **folds**: KFold shuffles the training months; TimeSeriesSplit keeps the order.
- **data leakage**: shuffled KFold can accidentally make the model look like it knows the future.
- **hyper parameters**: I tune `max_depth` and `min_samples_leaf`.
- **model selection**: the best tree is chosen inside each CV strategy.
- **independent test data set**: the last 20% of months are only used at the end.

My main result is that KFold gives a much higher internal score, but it does not hold up on the future test data. TimeSeriesSplit is less flashy, but it is the more honest setup for this problem.
